# MiniRocket Experiment

Runs MiniRocket on the six predefined aeon datasets, times feature extraction and classification, computes accuracy and macro F1, and saves the dataset-level results. Re-running replaces the matching MiniRocket rows for the same datasets and random state while retaining unrelated rows.

In [1]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
from aeon.datasets import load_classification
from aeon.transformations.collection.convolution_based import MiniRocket
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score, f1_score

### Cell 2 — Configuration and paths

Set the experiment parameters and resolve the repository root. Result files are written to `results/Aarti/`; the path logic works when the notebook is opened from the repository root or a subfolder.

In [2]:
DATASET_NAMES = ["Coffee", "ECG200", "GunPoint", "ItalyPowerDemand", "TwoLeadECG", "Wafer"]
RANDOM_STATE = 42
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (candidate for candidate in _candidates
     if (candidate / "results").is_dir()
     and (candidate / "hydra-authors-repo").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate the project root. Open this notebook from inside "
        "hydra-time-series-project, which must contain results/ and hydra-authors-repo/."
    )
RESULTS_DIR = PROJECT_ROOT / "results" / "Aarti"
RESULTS_FILE = RESULTS_DIR / "minirocket_results.csv"

### Cell 3 — Function

Define the per-dataset model pipeline: load train/test data, transform time series into features, fit the classifier, time each stage, calculate accuracy and macro F1, and return a result record.

In [3]:
def run_dataset(dataset_name):
    print(f"\n{'=' * 60}\nMiniRocket | {dataset_name}\n{'=' * 60}")
    X_train, y_train = load_classification(dataset_name, split="train")
    X_test, y_test = load_classification(dataset_name, split="test")
    model = MiniRocket(n_kernels=10000, max_dilations_per_kernel=32,
                       n_jobs=1, random_state=RANDOM_STATE)

    start = perf_counter()
    X_train_features = model.fit_transform(X_train)
    train_transform = perf_counter() - start

    start = perf_counter()
    X_test_features = model.transform(X_test)
    test_transform = perf_counter() - start

    classifier = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    start = perf_counter()
    classifier.fit(X_train_features, y_train)
    classifier_fit = perf_counter() - start

    start = perf_counter()
    y_pred = classifier.predict(X_test_features)
    prediction = perf_counter() - start

    row = {
        "dataset": dataset_name, "model": "MiniRocket",
        "random_state": RANDOM_STATE, "n_train": len(y_train), "n_test": len(y_test),
        "n_features": X_train_features.shape[1],
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "feature_extraction_train_time_seconds": train_transform,
        "classifier_train_time_seconds": classifier_fit,
        "total_training_time_seconds": train_transform + classifier_fit,
        "test_feature_extraction_time_seconds": test_transform,
        "prediction_time_seconds": prediction,
        "prediction_time_per_case_seconds": prediction / len(y_test),
        "end_to_end_test_time_seconds": test_transform + prediction,
    }
    print(f"Accuracy={row['accuracy']:.4f} | Macro F1={row['macro_f1']:.4f} | "
          f"Train={row['total_training_time_seconds']:.3f}s | "
          f"End-to-end test={row['end_to_end_test_time_seconds']:.3f}s")
    return row


### Cell 4 — Function

Define the top-level workflow: run each configured dataset, preserve unrelated existing rows, and save the updated model results to CSV.

In [4]:
def main():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    rows = []
    for name in DATASET_NAMES:
        try:
            rows.append(run_dataset(name))
        except Exception as exc:
            print(f"ERROR on {name}: {type(exc).__name__}: {exc}")
            raise
    new = pd.DataFrame(rows)
    if RESULTS_FILE.exists():
        old = pd.read_csv(RESULTS_FILE)
        old = old[~((old["dataset"].isin(DATASET_NAMES)) &
                    (old["random_state"] == RANDOM_STATE) &
                    (old["model"] == "MiniRocket"))]
        new = pd.concat([old, new], ignore_index=True)
    new.to_csv(RESULTS_FILE, index=False)
    print(f"\nSaved {len(rows)} dataset results to {RESULTS_FILE}")


### Cell 5 — Entry

Perform the next stage of the experiment or evaluation using the variables created in previous cells. Run cells in order because later cells depend on these definitions and results.

In [5]:
if __name__ == "__main__":
    main()


MiniRocket | Coffee
Accuracy=1.0000 | Macro F1=1.0000 | Train=0.809s | End-to-end test=0.022s

MiniRocket | ECG200
Accuracy=0.8900 | Macro F1=0.8783 | Train=0.095s | End-to-end test=0.025s

MiniRocket | GunPoint
Accuracy=0.9933 | Macro F1=0.9933 | Train=0.092s | End-to-end test=0.070s

MiniRocket | ItalyPowerDemand
Accuracy=0.9650 | Macro F1=0.9650 | Train=0.054s | End-to-end test=0.104s

MiniRocket | TwoLeadECG
Accuracy=0.9947 | Macro F1=0.9947 | Train=0.051s | End-to-end test=0.261s

MiniRocket | Wafer
Accuracy=0.9985 | Macro F1=0.9962 | Train=1.965s | End-to-end test=4.588s

Saved 6 dataset results to c:\Users\Aarti\Documents\GitHub\hydra-time-series-project\results\Aarti\minirocket_results.csv
